In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load CSV file
df = pd.read_csv("motorcycle_data_sample.csv")

# Display first rows
df.head()

,id,brand,model,year,variant_name,trim,category,subcategory,engine_type,cubic_capacity_cc,...,rear_brake,abs_type,traction_control,riding_modes,price_eur,features,is_electric,is_scooter,data_source,created_at
0,466,Aprilia,RS-GP,2021.0,Aprilia motorcycles Aprilia RS-GP 2021,NaN,sport,NaN,"V4, four-stroke",999.0,...,Single disc. Brembo calliper with two Ø 32 mm ...,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Liquid""...",False,False,bikez,2026-04-04T11:32:06.754747
1,343,BMW,450 Sports Enduro,2008.0,BMW BMW 450 Sports Enduro 2008,NaN,enduro,NaN,"Single cylinder, four-stroke",449.0,...,Single disc. Bremo,NaN,NaN,NaN,NaN,"[""Frame type: Lightweight stainless steel tube...",False,False,bikez,2026-04-04T11:32:06.754747
2,352,BMW,K1200R,2007.0,BMW BMW K1200R 2007,NaN,naked,NaN,"In-line four, four-stroke",1157.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Liquid""...",False,False,bikez,2026-04-04T11:32:06.754747
3,354,BMW,R 1200 Independent,2001.0,BMW BMW R 1200 Independent 2001,NaN,cruiser,NaN,"Two cylinder boxer, four-stroke",1170.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Air"", ""...",False,False,bikez,2026-04-04T11:32:06.754747
4,397,Ducati,Monster M750,1999.0,Ducati Ducati Monster M750 1999,NaN,naked,NaN,"V2, four-stroke",748.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Oil & a...",False,False,bikez,2026-04-04T11:32:06.754747


In [3]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   id                         30 non-null     int64  
 1   brand                      30 non-null     str    
 2   model                      30 non-null     str    
 3   year                       26 non-null     float64
 4   variant_name               30 non-null     str    
 5   trim                       0 non-null      float64
 6   category                   30 non-null     str    
 7   subcategory                7 non-null      str    
 8   engine_type                28 non-null     str    
 9   cubic_capacity_cc          29 non-null     float64
 10  cylinders                  0 non-null      float64
 11  cylinder_config            21 non-null     str    
 12  aspiration                 30 non-null     str    
 13  power_kw                   27 non-null     float64
 14  power_h

In [4]:
# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing text values
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].fillna("")

# Fill numerical missing values
numeric_columns = df.select_dtypes(include=np.number).columns

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

df.head()

,id,brand,model,year,variant_name,trim,category,subcategory,engine_type,cubic_capacity_cc,...,rear_brake,abs_type,traction_control,riding_modes,price_eur,features,is_electric,is_scooter,data_source,created_at
0,466,Aprilia,RS-GP,2021.0,Aprilia motorcycles Aprilia RS-GP 2021,NaN,sport,,"V4, four-stroke",999.0,...,Single disc. Brembo calliper with two Ø 32 mm ...,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Liquid""...",False,False,bikez,2026-04-04T11:32:06.754747
1,343,BMW,450 Sports Enduro,2008.0,BMW BMW 450 Sports Enduro 2008,NaN,enduro,,"Single cylinder, four-stroke",449.0,...,Single disc. Bremo,NaN,NaN,NaN,NaN,"[""Frame type: Lightweight stainless steel tube...",False,False,bikez,2026-04-04T11:32:06.754747
2,352,BMW,K1200R,2007.0,BMW BMW K1200R 2007,NaN,naked,,"In-line four, four-stroke",1157.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Liquid""...",False,False,bikez,2026-04-04T11:32:06.754747
3,354,BMW,R 1200 Independent,2001.0,BMW BMW R 1200 Independent 2001,NaN,cruiser,,"Two cylinder boxer, four-stroke",1170.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Air"", ""...",False,False,bikez,2026-04-04T11:32:06.754747
4,397,Ducati,Monster M750,1999.0,Ducati Ducati Monster M750 1999,NaN,naked,,"V2, four-stroke",748.0,...,Single disc,NaN,NaN,NaN,NaN,"[""Starter: Electric"", ""Cooling system: Oil & a...",False,False,bikez,2026-04-04T11:32:06.754747


In [5]:
text_columns = df.select_dtypes(include="object").columns

print("Text columns:")
print(text_columns)

Text columns:
Index(['brand', 'model', 'variant_name', 'category', 'subcategory',
       'engine_type', 'cylinder_config', 'aspiration', 'transmission_type',
       'fuel_type', 'front_brake', 'rear_brake', 'features', 'data_source',
       'created_at'],
      dtype='str')


In [6]:
# Combine all text fields into one column

df["combined_text"] = df[text_columns].astype(str).apply(
    lambda x: " ".join(x),
    axis=1
)

df["combined_text"].head()

0    Aprilia RS-GP Aprilia motorcycles Aprilia RS-G...
1    BMW 450 Sports Enduro BMW BMW 450 Sports Endur...
2    BMW K1200R BMW BMW K1200R 2007 naked  In-line ...
3    BMW R 1200 Independent BMW BMW R 1200 Independ...
4    Ducati Monster M750 Ducati Ducati Monster M750...
Name: combined_text, dtype: str

In [7]:
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=500
)

text_features = tfidf.fit_transform(
    df["combined_text"]
)

text_features.shape

(30, 354)

In [8]:
numeric_features = df.select_dtypes(
    include=np.number
)

numeric_features.head()

,id,year,trim,cubic_capacity_cc,cylinders,power_kw,power_hp,torque_nm,gears,drive_type,...,length_mm,width_mm,height_mm,wheelbase_mm,front_suspension,rear_suspension,abs_type,traction_control,riding_modes,price_eur
0,466,2021.0,NaN,999.0,NaN,186.1,255.0,115.0,6.0,NaN,...,2040.0,820.0,1125.0,1420.0,NaN,NaN,NaN,NaN,NaN,NaN
1,343,2008.0,NaN,449.0,NaN,36.2,49.6,48.0,5.0,NaN,...,2189.0,820.0,1125.0,1460.5,NaN,NaN,NaN,NaN,NaN,NaN
2,352,2007.0,NaN,1157.0,NaN,119.0,163.0,127.0,6.0,NaN,...,2228.0,820.0,1125.0,1580.0,NaN,NaN,NaN,NaN,NaN,NaN
3,354,2001.0,NaN,1170.0,NaN,44.5,61.0,98.0,5.0,NaN,...,2189.0,820.0,1125.0,1650.0,NaN,NaN,NaN,NaN,NaN,NaN
4,397,1999.0,NaN,748.0,NaN,46.7,64.0,62.0,5.0,NaN,...,2189.0,820.0,1125.0,1430.0,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(
    numeric_features
)

scaled_numeric.shape

(30, 27)

In [10]:
df.to_csv(
    "motorcycle_clusters_output.csv",
    index=False
)

print("File saved successfully")

File saved successfully
